# Milestone 4 - Multiple Choice Model Setup


## Setup


In [1]:
!unzip smart-mcq-solver-challenge.zip


Archive:  smart-mcq-solver-challenge.zip
  inflating: sample_submission.csv   
  inflating: test.csv                
  inflating: train.csv               


In [2]:
!pip install transformers peft datasets -q


In [3]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForMultipleChoice, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("using device:", device)

data = pd.read_csv("train.csv")
data.head()


using device: cuda


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


## Q1. Label Encoding

Map the answer column to numbers, A to 0, B to 1, and so on, then check row 150.


In [4]:
choice_to_idx = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}
data["label"] = data["answer"].apply(lambda ans: choice_to_idx[ans])

label_at_150 = data.at[150, "label"]
print("encoded label at index 150:", label_at_150)


encoded label at index 150: 2


## Q2. Prompt-Option Formatting

Build the Option B input for row 0 and check its length.


In [5]:
first_row = data.loc[0]
b_option_text = f'{first_row["prompt"]} [SEP] {first_row["B"]}'

print(b_option_text)
print("length:", len(b_option_text))


Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options. [SEP] Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement.
length: 407


## Tokenization for Multiple-Choice Models

Each row has 5 options, so each row turns into 5 tokenized sequences.
We load the tokenizer once and reuse it below.


In [6]:
mcq_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

## Q3. Single-Row MCQ Tokenization

Tokenize the 5 options for row 0 and reshape into [1, 5, 128].


In [7]:
option_letters = ["A", "B", "C", "D", "E"]

prompt_text = first_row["prompt"]
option_texts = [f"{prompt_text} [SEP] {first_row[letter]}" for letter in option_letters]

row0_encoding = mcq_tokenizer(
    option_texts,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

row0_input_ids = row0_encoding["input_ids"].unsqueeze(0)  # [1, 5, 128]
print("input_ids shape:", row0_input_ids.shape)
print("second dimension:", row0_input_ids.shape[1])


input_ids shape: torch.Size([1, 5, 128])
second dimension: 5


## Q4. Batch MCQ Tokenization

Do the same thing for the first 16 rows.


In [8]:
first_16 = data.iloc[:16]

nested_choice_texts = []
for _, row in first_16.iterrows():
    p_text = row["prompt"]
    nested_choice_texts.append([f"{p_text} [SEP] {row[letter]}" for letter in option_letters])

flattened_choices = [text for row_texts in nested_choice_texts for text in row_texts]

batch_encoding = mcq_tokenizer(
    flattened_choices,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

batch_input_ids = batch_encoding["input_ids"].view(16, 5, 128)
print("batch input_ids shape:", batch_input_ids.shape)

n_rows, n_choices, seq_len = batch_input_ids.shape
total_token_positions = n_rows * n_choices * seq_len
print("total token positions:", total_token_positions)


batch input_ids shape: torch.Size([16, 5, 128])
total token positions: 10240


## Multiple-Choice Model Outputs

The model gives one logit per option, so 5 logits per question.


In [9]:
mcq_model = AutoModelForMultipleChoice.from_pretrained("bert-base-uncased")
mcq_model.to(device)


model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BertForMultipleChoice(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, element

## Q5. Multiple-Choice Logits

Run row 0 through the model and check the logits shape.


In [10]:
mcq_model.eval()

row0_ids_on_device = row0_input_ids.to(device)
row0_mask_on_device = row0_encoding["attention_mask"].unsqueeze(0).to(device)

with torch.no_grad():
    row0_output = mcq_model(input_ids=row0_ids_on_device, attention_mask=row0_mask_on_device)

print("logits shape:", row0_output.logits.shape)
print("logits for one question:", row0_output.logits.shape[1])


logits shape: torch.Size([1, 5])
logits for one question: 5


## Q6. Supervised Loss Tensor

Pass in the correct label along with the input and check the loss tensor.


In [11]:
row0_label = torch.tensor([data.at[0, "label"]]).to(device)

with torch.no_grad():
    row0_output_with_loss = mcq_model(
        input_ids=row0_ids_on_device,
        attention_mask=row0_mask_on_device,
        labels=row0_label
    )

print("loss value:", row0_output_with_loss.loss.item())
print("loss tensor dims:", row0_output_with_loss.loss.dim())


loss value: 1.6168476343154907
loss tensor dims: 0


## LoRA for Efficient Fine-Tuning

LoRA freezes the base model and only trains small adapter matrices added to a
few target modules. Here we target the query and value projections.


## Q7. LoRA Trainable Parameters


In [12]:
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS
)

# Install a version of torchao that meets the requirement (>0.16.0)
!pip install torchao==0.17.0

peft_model = get_peft_model(mcq_model, peft_config)
peft_model.to(device)

n_trainable = 0
for param in peft_model.parameters():
    if param.requires_grad:
        n_trainable += param.numel()

print("trainable parameters:", n_trainable)
peft_model.print_trainable_parameters()


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 45.5 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


trainable parameters: 295681
trainable params: 295,681 || all params: 109,778,690 || trainable%: 0.2693


## Preparing Data for Hugging Face Trainer

Turn the first 100 rows into a Hugging Face Dataset with input_ids,
attention_mask, and labels ready for training.


## Q8. Hugging Face Dataset Preparation


In [13]:
def row_to_features(row, seq_len=128):
    p_text = row["prompt"]
    texts = [f"{p_text} [SEP] {row[letter]}" for letter in option_letters]
    tokenized = mcq_tokenizer(
        texts,
        padding="max_length",
        truncation=True,
        max_length=seq_len,
        return_tensors="pt"
    )
    return {
        "input_ids": tokenized["input_ids"].tolist(),
        "attention_mask": tokenized["attention_mask"].tolist(),
        "labels": row["label"]
    }

hundred_rows = data.iloc[:100].reset_index(drop=True)
feature_dicts = [row_to_features(row) for _, row in hundred_rows.iterrows()]

train_dataset_100 = Dataset.from_list(feature_dicts)

sample_item = train_dataset_100[0]
sample_ids_tensor = torch.tensor(sample_item["input_ids"])
print("input_ids shape:", sample_ids_tensor.shape)
print("number of choices stored:", sample_ids_tensor.shape[0])


input_ids shape: torch.Size([5, 128])
number of choices stored: 5


## Tiny Fine-Tuning and Inference

Train the LoRA model on a small slice of data for just a few steps, then
run inference on row 0.


## Q9. Tiny LoRA Fine-Tuning


In [14]:
def row_to_features_short(row, seq_len=64):
    p_text = row["prompt"]
    texts = [f"{p_text} [SEP] {row[letter]}" for letter in option_letters]
    tokenized = mcq_tokenizer(
        texts,
        padding="max_length",
        truncation=True,
        max_length=seq_len,
        return_tensors="pt"
    )
    return {
        "input_ids": tokenized["input_ids"].tolist(),
        "attention_mask": tokenized["attention_mask"].tolist(),
        "labels": row["label"]
    }

thirty_two_rows = data.iloc[:32].reset_index(drop=True)
short_feature_dicts = [row_to_features_short(row) for _, row in thirty_two_rows.iterrows()]
tiny_train_dataset = Dataset.from_list(short_feature_dicts)

lora_training_args = TrainingArguments(
    output_dir="mcq_lora_run_variant_b",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    max_steps=4,
    logging_steps=1,
    report_to="none"
)

lora_trainer = Trainer(
    model=peft_model,
    args=lora_training_args,
    train_dataset=tiny_train_dataset
)

lora_train_result = lora_trainer.train()
print("final global_step:", lora_trainer.state.global_step)


[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
1,1.686629
2,1.626602
3,1.612082
4,1.598551


final global_step: 4


## Q10. Probability Assigned to Option E After Fine-Tuning


In [15]:
peft_model.eval()

# The model already lives on `device` from earlier; reuse the same tensors
with torch.no_grad():
    tuned_output = peft_model(input_ids=row0_ids_on_device, attention_mask=row0_mask_on_device)

option_probs = torch.softmax(tuned_output.logits, dim=1)
prob_option_e = option_probs[0, 4].item()

print("probabilities:", option_probs)
print("probability for option E:", round(prob_option_e, 4))


probabilities: tensor([[0.2015, 0.1986, 0.1976, 0.1998, 0.2024]], device='cuda:0')
probability for option E: 0.2024
